# Data Exploration Report - for "RAGuardianNews" Prototype
IDS 570 - Text as Data\
Peter de Guzman ped19\
02/26/26

# Table of Contents
* [Project Overview](#project-overview)
    * [Description of Data](#description-of-data)
* [Data Pipeline and Application Functionality](#data-pipeline-and-application-functionality)
    * [Data Pre-processing Procedures](#data-pre-processing-procedures)
* [Metadata Analysis](#metadata-analysis)
* [Measures of Lexican Distinctiveness, Similarity, and Syntactic Complexity](#measures-of-lexical-distinctiveness-similarity-and-syntactic-complexity)
    * [TF-IDF Analysis](#tf-idf-analysis)
    * [Pearson Correlation](#pearson-correlation)
    * [Syntactic Complexity Measures](#syntactic-complexity-measures)
* [Synthesis of Findings](#synthesis-of-findings)
* [Appendix](#appendix)

<h2 id="project-overview">Project Overview</h2>

This project aims to perform Named Entity Recognition (NER) and leverage Retrieval Augmented Generation (RAG) to allow users to easily query articles from **The Guardian** (a British daily newspaper) regarding artificial intelligence using natural language. Users will be able to enter in a question or topic and the system will find relevant articles and summarize key information or answer the specific query using large language models (LLMs).

Note: This report will focus only on the initial steps of data collection (Extract, Transform, and Load) and a meta-analysis of the dataset.

<h3 id="description-of-data">Description of Data</h2>

For this project, news articles were downloaded from the Guardian API for the time period of `2016-02-22` to `2026-02-22`. The 10 year period was chosen for this pilot project to collect a large number of articles while still avoiding rate limitations when downloading articles through the API. The 10 year period allowed for the inclusion of articles from before the release of GPT-1 & BERT in 2018, and the widespread adoption of ChatGPT between 2020-2022. The query terms used for the API keyword search were "artificial intelligence", "AI", "generative AI", and "GenAI". These search terms were chosen for this prototype to capture the rice in AI technologies while being more narrow than simply downloading articles related to "technology". The 'q' query parameter used with the API ensures that the API call only returns content that explicitly includes the keyword search term. 

The resulting dataset is 24,124 articles, which was downloaded from the API and saved in a JSON file format. 

<h2 id="data-pipeline">Data Pipeline</h2>


<h3 id="data-pipeline-and-application-functionality">Data Pipeline and Application Functionality</h2>

```mermaid
flowchart LR
    %% Data ingestion
    API["API"]:::ingest --> DB_RAW["DuckDB (Raw Articles)"]:::storage

    %% Transformation with dbt
    DB_RAW -->|dbt run| DB_CLEAN["DuckDB (Cleaned Articles)"]:::storage

    %% NLP / downstream
    DB_CLEAN --> NER["NER Extraction"]:::process
    NER --> RAG["RAG Embeddings / QA"]:::process

    %% Final query
    RAG --> LLM["LLM Query"]:::query

    %% Feedback loops
    LLM -- Feedback / Updates --> RAG
    LLM -- Feedback / Updates --> NER

    %% Style definitions
    classDef ingest fill:#a2d2ff,stroke:#0d3b66,stroke-width:1px,color:#0d3b66
    classDef storage fill:#ffb703,stroke:#fb8500,stroke-width:1px,color:#000000
    classDef process fill:#8ac926,stroke:#3f681c,stroke-width:1px,color:#000000
    classDef query fill:#9d4edd,stroke:#6a0dad,stroke-width:1px,color:#ffffff
```

<h3 id="data-pre-processing-procedures">Data Pre-processing Procedures</h3>

Data Transformation with `dbt`:

Once articles were downloaded, they were de-duplicated by the "id". Because I did not implement a ranking algorithm for this de-duplication, if the same article was downloaded under multiple search terms, the last search term was retained. This is a limitation to this approach because the resulting dataset was heavily imbalanced and 99% of the retained articles were assigned search terms "artificial intelligence" or "generative AI".. It is possible that there was overlap in articles and search terms, but this process was chosen due to timeline and rate limitation constraints. An improvement to this analysis would incorporate a method for accounting for articles that appeared under multiple search terms and flag this attribute, while retaining the maximum amount of information per article. 

From the JSON file, articles are loaded into a DuckDB database and saved in the `raw_articles` table. To clean the body text of the articles and prepare them for analysis, the `cleaned_articles.sql` dbt model was written which points to the `raw_articles` table and removes all HTML tags from the body text column before saving it as 'clean_body'. I decided to not remove the time stamps (such as '8.31 BST') that were observed because some articles that possess these time stamps are live updates while others represent back and forth dialogues. Therefore, the time stamps provide information regarding the structure of the article and possess inherent meaning. It is possible that at a later date it may be helpful to remove these time stamps from the cleaned dataset if they are adding additional noise to our analysis and are not useful. 

Tests are also implemented with `dbt` to check that all values for id and the body text are unique and not null in the `cleaned_articles` and `sample_articles` tables. These tests can be implemented by navigating to the `dbt_guardian/` folder and running the `uv run dbt test` command. 

<h2 id="metadata-analysis">Metadata Analysis</h2>


Once the data is extracted from the Guardian API, loaded into the DuckDB database and transformed into a cleaned dataset, we are able to analyze the metadata to review its completeness over the years and trends over time by its various features. 

The columns present in the cleaned table include: ['id', 'type', 'sectionId', 'sectionName', 'webPublicationDate', 'webTitle', 'webUrl', 'apiUrl', 'body', 'isHosted', 'pillarId', 'pillarName', 'headline', 'shortUrl', 'search_term', 'pull_date', 'clean_body'].
- The API call includes a parameter "isHosted" which is "false" if the article is Regular Guardian editorial content that is produced by their newsroom and "true" if it is sponsored content or content produced in partnership with another organization. None of the articles in this dataset were marked as sponsored content. 
- The "pillar" classification system is a topical distinction in articles, with categories such as "News", "Opinion", "Sports", "Arts", "Lifestyle". The distribution of these topics is displayed below.
- The "section" classification denotes as specific desk that the article was published under, and is more granular. The distribution of these sections is also displayed below. 


<h2 id="measures-of-lexical-distinctiveness-similarity-and-syntactic-complexity">Measures of Lexical Distinctiveness, Similarity, and Syntactic Complexity</h2>

<h3 id="tf-idf-analysis">TF-IDF Analysis</h3>


<h3 id="pearson-correlation">Pearson Correlation</h3>


<h3 id="syntactic-complexity-measures">Syntactic Complexity Measures</h3>


- talk about top five articles by MLS
- these seem to be the same as the top five articles with clauses per sentence, obivously because they have the longest sentences
- dependent clause per clause, dependent clause per sentence, coord per sentence
- complex nominals


<h2 id="synthesis-of-findings">Synthesis of Findings</h2>


<h3 id="limitations-and-future-work">Limitations and Future Work</h3>

This analysis represents a limited exploration into the Guardian's overall corpus of news articles and publicly available content. The term "artificial intelligence" was first used in the early 1950s by the computer scientist John McCarthy and his colleagues at the Dartmouth Conference (Haenlein & Kaplan 2019). This analysis was limited to querying the Guardian API tool with the search terms "artificial intelligence" and "generative AI", although it is likely that some articles may have been missed that contain discussion of these and similar technologies but did not explicitly contain these keyword search terms. 

- only looked at a random sample of the large corpus, sampled 10 articles for each search term, there may be different findings when comparing articles by different qualities (authors, year published, sectionName). 
- Although some this analysis compared findings by search term, it is not yet clear that the decision to use "artificial intelligence" versus "generative AI" when writing about these technologies is accompanied by an intent to tell a different narrative or approach these technologies from a different paradigm. 


Future Direction:
- implement NER with BERT
- alow users to query the corpus by different features
- allow for RAG with LLM model so users can ask questions about artificial intelligence using natural language

<h3 id="appendix">Appendix</h3>


Guardian API Documentation: https://open-platform.theguardian.com/documentation/

References
https://bpb-us-w2.wpmucdn.com/sites.uab.edu/dist/6/536/files/2024/08/A-brief-history-of-AI.pdf